# 05 · Two-hop evidence chain (Prompt 11), owner: M3

Before running:
1. **Accelerator: GPU T4 x1** (CPU also works, but HotpotQA encoding is slower). **Internet: On**.
2. **Add Input** (Your Work):
   - `fever-corpus-v1` (Datasets): the corpus with hyperlinks per sentence, the splits, and
     `raw/hotpot_dev_distractor_v1.json`
   - the output of notebook **02 · BM25**
   - the output of notebook **04 · Fine-tune + hybrid**: `models/minilm_ft`, `outputs/dense_ft/embeddings`,
     `outputs/hybrid`, `outputs/rerank`

Expected time: about 10-20 minutes.

When it finishes: **Save Version → Save & Run All (Commit)**. Notebook 06 needs `outputs/chain/`.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/group-mauve/claim-verification-pipeline.git"   # public repo
BRANCH = "main"
WORK = "/kaggle/working/project"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError("git failed (see message above). Is Internet on in the notebook settings?")

if not os.path.exists(WORK):
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK)
else:
    git("-C", WORK, "pull", "origin", BRANCH)   # pick up teammates' latest pushes
os.chdir(WORK)
git("log", "--oneline", "-1")                    # which commit this run uses
print(os.getcwd(), sorted(os.listdir()))

In [ ]:
!pip install -q faiss-cpu sentence-transformers spacy pyyaml pytest
!python -m spacy download en_core_web_sm || echo "spaCy model download failed: the chain falls back to capitalised words"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Which inputs are attached? Every folder this notebook needs should appear in this list.
import glob
found = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True)
               if p.endswith(("sentences.parquet", "metrics.json", "minilm_ft/config.json", "hotpot_dev_distractor_v1.json")))
print("\n".join(found) if found else "Nothing attached: use Add Input (see the first cell)")

## Unit tests

In [ ]:
!python -m pytest -q -rs

## Prompt 11 · Build chains, tune the hop-2 threshold on validation, evaluate on FEVER and HotpotQA

In [ ]:
!python -m src.chain.two_hop --config configs/chain.yaml

In [ ]:
import json, pandas as pd
m = json.load(open("outputs/chain/metrics.json"))
print(f"hop 1 from: {m['hop1_run']} | threshold {m['threshold']} | missing_link on test: {m['fever']['missing_link_share_test']:.1%}")
rows = {}
for subset in ("multi_page", "all_sr"):
    for hop in ("one_hop", "two_hop"):
        r = m["fever"][subset][hop]
        rows[(f"FEVER {subset} ({r['n_claims']:,} claims)", hop)] = {k: r[k] for k in ("sf_precision", "sf_recall", "sf_f1", "complete_set_rate")}
if m["hotpot"]:
    for hop in ("one_hop", "two_hop"):
        rows[(f"HotpotQA ({m['hotpot']['n_eval']:,} questions)", hop)] = m["hotpot"][hop]
pd.DataFrame(rows).T.round(4)

In [ ]:
pd.DataFrame(m["threshold_grid"])[["threshold", "complete_set_rate", "sf_recall", "sf_f1", "share_with_hop2"]].round(4)

In [ ]:
for ex in json.load(open("outputs/chain/examples.json")):
    print(f"--- {ex['kind']}: [{ex['label']}] {ex['claim']} | bridges {[b['entity'] for b in ex['bridges']]}")
    for it in ex["chain"]:
        print(f"   hop {it['hop']} ({it['page_id']}, {it['sent_id']}) {it['score']:.3f} via {it['bridge']} | {it['text'][:110]}")

In [ ]:
!du -sh outputs/*